# Step 5: ChronoBERT Embeddings (Colab GPU)

**Instructions:**
1. Set runtime to **GPU**: Runtime → Change runtime type → T4 GPU
2. Run all cells — it will prompt you to upload `sections.parquet` and `deals.parquet`
3. Download the output files at the end

Estimated time: ~15-30 min depending on GPU.

In [ ]:
!pip install -q transformers torch pyyaml pandas pyarrow

In [ ]:
# Upload your data files
from google.colab import files
import os

print("Upload sections.parquet and deals.parquet")
uploaded = files.upload()
print(f"Uploaded: {list(uploaded.keys())}")

In [ ]:
import numpy as np
import pandas as pd
import torch
import gc
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ── Config ────────────────────────────────────────────────────────────────
MODEL_FAMILY = "manelalab/chrono-bert-v1-{y}1231"
LAG = 1           # vintage = announce_year - 1
MAX_TOKENS = 512  # reduced from 1024 to save memory
BATCH_SIZE = 8    # reduced from 32 to save memory

# ── Load data ─────────────────────────────────────────────────────────────
sections = pd.read_parquet("sections.parquet")
deals = pd.read_parquet("deals.parquet")

# Keep only pre-merger sections
sections = sections[sections["purpose"] == "pre"].copy()
print(f"Pre-merger paragraphs: {len(sections)} across {sections['dealnum'].nunique()} deals")

# Map dealnum -> announce year
deal_year = deals.set_index("dealnum")["ayear"].astype(float).astype(int).to_dict()
sections["ayear"] = sections["dealnum"].map(deal_year)
sections = sections.dropna(subset=["ayear"])
sections["ayear"] = sections["ayear"].astype(int)
print(f"Vintage years: {sorted(sections['ayear'].unique())}")

In [ ]:
import pickle

@torch.no_grad()
def embed_paragraphs(paragraphs, tokenizer, model):
    """Encode paragraphs and return mean-pooled 768-dim vector."""
    if not paragraphs:
        return np.zeros(768, dtype=np.float32)

    all_cls = []
    for i in range(0, len(paragraphs), BATCH_SIZE):
        batch = paragraphs[i : i + BATCH_SIZE]
        enc = tokenizer(
            batch, padding=True, truncation=True,
            max_length=MAX_TOKENS, return_tensors="pt",
        ).to(device)
        out = model(**enc)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_cls.append(cls)
        # Free GPU memory
        del enc, out
        torch.cuda.empty_cache()

    return np.concatenate(all_cls, axis=0).mean(axis=0).astype(np.float32)


# ── Embed by vintage year (with checkpointing) ───────────────────────────
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load any previously checkpointed results
results = []
completed_years = set()
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    if f.startswith("vintage_") and f.endswith(".pkl"):
        year = int(f.split("_")[1].split(".")[0])
        with open(os.path.join(CHECKPOINT_DIR, f), "rb") as fh:
            year_results = pickle.load(fh)
        results.extend(year_results)
        completed_years.add(year)
        print(f"Loaded checkpoint for vintage {year}: {len(year_results)} groups")

for ayear in sorted(sections["ayear"].unique()):
    if ayear in completed_years:
        print(f"\nSkipping vintage {ayear} (already checkpointed)")
        continue

    repo = MODEL_FAMILY.format(y=ayear - LAG)
    print(f"\n{'='*60}")
    print(f"Vintage {ayear} -> {repo}")

    # Load model fresh each vintage, free previous
    print("  Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(repo)
    model = AutoModel.from_pretrained(repo).to(device).eval()

    year_df = sections[sections["ayear"] == ayear]
    groups = list(year_df.groupby(["dealnum", "role", "item"]))
    n_groups = len(groups)
    print(f"  {n_groups} groups to embed")

    year_results = []
    for idx, ((dealnum, role, item), grp) in enumerate(groups):
        if (idx + 1) % 10 == 0 or idx == 0:
            print(f"  {idx + 1}/{n_groups} embeddings...")

        paragraphs = grp["text"].tolist()
        emb = embed_paragraphs(paragraphs, tokenizer, model)

        year_results.append({
            "dealnum": dealnum,
            "role": role,
            "item": item,
            "n_paragraphs": len(paragraphs),
            "embedding": emb,
        })

    # Checkpoint this vintage to disk
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"vintage_{ayear}.pkl")
    with open(ckpt_path, "wb") as fh:
        pickle.dump(year_results, fh)
    print(f"  Saved checkpoint: {ckpt_path}")

    results.extend(year_results)

    # Free model memory before loading next vintage
    del tokenizer, model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Done with {ayear}, freed GPU memory")

print(f"\nDone! Total embeddings: {len(results)}")

In [ ]:
# ── Pivot to deal-level and save ──────────────────────────────────────────
emb_df = pd.DataFrame(results)

rows = []
for dealnum, grp in emb_df.groupby("dealnum"):
    row = {"dealnum": dealnum}
    for _, r in grp.iterrows():
        prefix = f"{r['role']}_{r['item']}"
        row[prefix] = r["embedding"]
        row[f"{prefix}_n"] = r["n_paragraphs"]
    rows.append(row)

deal_emb = pd.DataFrame(rows)
print(f"Deal-level embeddings: {len(deal_emb)} deals")

# Save embeddings
os.makedirs("embeddings", exist_ok=True)

emb_cols = [c for c in deal_emb.columns if c.endswith("_item1") or c.endswith("_item7")]
np.savez(
    "embeddings/embeddings.npz",
    dealnums=deal_emb["dealnum"].values,
    **{col: np.stack(deal_emb[col].dropna().values) for col in emb_cols},
)

meta_cols = ["dealnum"] + [c for c in deal_emb.columns if c.endswith("_n")]
deal_emb[meta_cols].to_parquet("embeddings/metadata.parquet", index=False)

print("Saved embeddings/embeddings.npz and embeddings/metadata.parquet")

In [ ]:
# Download the output files
from google.colab import files

files.download("embeddings/embeddings.npz")
files.download("embeddings/metadata.parquet")
print("Done! Place these in data/processed/embeddings/ in your local project.")